In [9]:
# Imports
import json
import concurrent.futures
import re
from textwrap import dedent
from statistics import mean
from dotenv import load_dotenv
from anthropic import Anthropic

In [10]:
# Client Initialization and helper functions

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"


def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [11]:
# Report Builder
def generate_prompt_evaluation_report(evaluation_results):
    total_tests = len(evaluation_results)
    scores = [result["score"] for result in evaluation_results]
    avg_score = mean(scores) if scores else 0
    max_possible_score = 10
    pass_rate = (
        100 * len([s for s in scores if s >= 7]) / total_tests if total_tests else 0
    )

    html = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Prompt Evaluation Report</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                line-height: 1.6;
                margin: 0;
                padding: 20px;
                color: #333;
            }}
            .header {{
                background-color: #f0f0f0;
                padding: 20px;
                border-radius: 5px;
                margin-bottom: 20px;
            }}
            .summary-stats {{
                display: flex;
                justify-content: space-between;
                flex-wrap: wrap;
                gap: 10px;
            }}
            .stat-box {{
                background-color: #fff;
                border-radius: 5px;
                padding: 15px;
                box-shadow: 0 2px 5px rgba(0,0,0,0.1);
                flex-basis: 30%;
                min-width: 200px;
            }}
            .stat-value {{
                font-size: 24px;
                font-weight: bold;
                margin-top: 5px;
            }}
            table {{
                width: 100%;
                border-collapse: collapse;
                margin-top: 20px;
            }}
            th {{
                background-color: #4a4a4a;
                color: white;
                text-align: left;
                padding: 12px;
            }}
            td {{
                padding: 10px;
                border-bottom: 1px solid #ddd;
                vertical-align: top;
            }}
            tr:nth-child(even) {{
                background-color: #f9f9f9;
            }}
            .output-cell {{
                white-space: pre-wrap;
            }}
            .score {{
                font-weight: bold;
                padding: 5px 10px;
                border-radius: 3px;
                display: inline-block;
            }}
            .score-high {{
                background-color: #c8e6c9;
                color: #2e7d32;
            }}
            .score-medium {{
                background-color: #fff9c4;
                color: #f57f17;
            }}
            .score-low {{
                background-color: #ffcdd2;
                color: #c62828;
            }}
            .output {{
                overflow: auto;
                white-space: pre-wrap;
            }}

            .output pre {{
                background-color: #f5f5f5;
                border: 1px solid #ddd;
                border-radius: 4px;
                padding: 10px;
                margin: 0;
                font-family: 'Consolas', 'Monaco', 'Courier New', monospace;
                font-size: 14px;
                line-height: 1.4;
                color: #333;
                box-shadow: inset 0 1px 3px rgba(0, 0, 0, 0.1);
                overflow-x: auto;
                white-space: pre-wrap; 
                word-wrap: break-word; 
            }}

            td {{
                width: 20%;
            }}
            .score-col {{
                width: 80px;
            }}
        </style>
    </head>
    <body>
        <div class="header">
            <h1>Prompt Evaluation Report</h1>
            <div class="summary-stats">
                <div class="stat-box">
                    <div>Total Test Cases</div>
                    <div class="stat-value">{total_tests}</div>
                </div>
                <div class="stat-box">
                    <div>Average Score</div>
                    <div class="stat-value">{avg_score:.1f} / {max_possible_score}</div>
                </div>
                <div class="stat-box">
                    <div>Pass Rate (≥7)</div>
                    <div class="stat-value">{pass_rate:.1f}%</div>
                </div>
            </div>
        </div>

        <table>
            <thead>
                <tr>
                    <th>Scenario</th>
                    <th>Prompt Inputs</th>
                    <th>Solution Criteria</th>
                    <th>Output</th>
                    <th>Score</th>
                    <th>Reasoning</th>
                </tr>
            </thead>
            <tbody>
    """

    for result in evaluation_results:
        prompt_inputs_html = "<br>".join(
            [
                f"<strong>{key}:</strong> {value}"
                for key, value in result["test_case"]["prompt_inputs"].items()
            ]
        )

        criteria_string = "<br>• ".join(result["test_case"]["solution_criteria"])

        score = result["score"]
        if score >= 8:
            score_class = "score-high"
        elif score <= 5:
            score_class = "score-low"
        else:
            score_class = "score-medium"

        html += f"""
            <tr>
                <td>{result["test_case"]["scenario"]}</td>
                <td class="prompt-inputs">{prompt_inputs_html}</td>
                <td class="criteria">• {criteria_string}</td>
                <td class="output"><pre>{result["output"]}</pre></td>
                <td class="score-col"><span class="score {score_class}">{score}</span></td>
                <td class="reasoning">{result["reasoning"]}</td>
            </tr>
        """

    html += """
            </tbody>
        </table>
    </body>
    </html>
    """

    return html

In [12]:
# PromptEvaluator Implementation
class PromptEvaluator:
    def __init__(self, max_concurrent_tasks=3):
        self.max_concurrent_tasks = max_concurrent_tasks

    def render(self, template_string, variables):
        placeholders = re.findall(r"{([^{}]+)}", template_string)

        result = template_string
        for placeholder in placeholders:
            if placeholder in variables:
                result = result.replace(
                    "{" + placeholder + "}", str(variables[placeholder])
                )

        return result.replace("{{", "{").replace("}}", "}")

    def generate_unique_ideas(self, task_description, prompt_inputs_spec, num_cases):
        """Generate a list of unique ideas for test cases based on the task description"""

        prompt = """
       Gere {num_cases} ideias únicas e diversas para testar um prompt que realize esta tarefa:

        <task_description>
        {task_description}
        </task_description>
        
        O prompt receberá as seguintes entradas
        <prompt_inputs>
        {prompt_inputs_spec}
        </prompt_inputs>

        Cada ideia deve representar um cenário ou exemplo distinto que teste diferentes aspectos da tarefa.

        Formato de saída:
        Forneça sua resposta como um array JSON estruturado, em que cada item seja uma breve descrição da ideia.

        Exemplo:
        ```json
        [
            "Teste com terminologia técnica de ciência da computação"
            "Teste com descobertas de pesquisas médicas",
            "Teste com conceitos matemáticos complexos",
            ...
        ]
        ```

        Garanta que cada ideia seja:
        - Claramente distinta das outras
        - Relevante para a descrição da tarefa
        - Específica o suficiente para orientar a geração de um caso de teste completo
        - Rápida de resolver, sem exigir computação extensa ou processamento em múltiplas etapas
        - Solucionável com no máximo 400 tokens de saída

        Lembre-se: gere apenas {num_cases} ideias únicas.
        """

        system_prompt = "Você é um designer de cenários de teste especializado em criar cenários de teste diversos e únicos."

        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = value.replace("\n", "\\n")
            example_prompt_inputs += f'"{key}": str # {val},'

        rendered_prompt = self.render(
            dedent(prompt),
            {
                "task_description": task_description,
                "num_cases": num_cases,
                "prompt_inputs": example_prompt_inputs,
            },
        )

        messages = []
        add_user_message(messages, rendered_prompt)
        add_assistant_message(messages, "```json")
        text = chat(
            messages,
            stop_sequences=["```"],
            system=system_prompt,
            temperature=1.0,
        )

        return json.loads(text)

    def generate_test_case(self, task_description, idea, prompt_inputs_spec={}):
        """Generate a single test case based on the task description and a specific idea"""

        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = value.replace("\n", "\\n")
            example_prompt_inputs += f'"{key}": "EXAMPLE_VALUE", // {val}\n'

        allowed_keys = ", ".join([f'"{key}"' for key in prompt_inputs_spec.keys()])

        prompt = """
        Gere um único caso de teste detalhado para uma avaliação de prompt com base em:
        
        <task_description>
        {task_description}
        </task_description>
        
        <specific_idea>
        {idea}
        </specific_idea>
        
        <allowed_input_keys>
        {allowed_keys}
        </allowed_input_keys>
        
        Output Format:
        ```json
        {{
            "prompt_inputs": {{
            {example_prompt_inputs}
            }},
            "solution_criteria": ["criterion 1", "criterion 2", ...] // Lista concisa de critérios para avaliar a solução, de 1 a 4 itens
        }}
        ```
        
        REQUISITOS IMPORTANTES:
        - Você DEVE USAR APENAS essas chaves de entrada exatas em seus prompt_inputs: {allowed_keys}        
        - NÃO adicione nenhuma chave adicional aos prompt_inputs
        - Todas as chaves listadas em allowed_input_keys devem ser incluídas em sua resposta
        - Faça o caso de teste realista e praticamente útil
        - Inclua critérios de solução mensuráveis e concisos
        - Os critérios de solução devem APENAS abordar os requisitos diretos da descrição da tarefa e dos prompt_inputs gerados
        - Evite especificar demais os critérios com requisitos que vão além da tarefa principal
        - Mantenha os critérios de solução simples, focados e diretamente ligados à tarefa fundamental
        - O caso de teste deve ser adaptado à ideia específica fornecida
        - Rápido de resolver, sem exigir computação extensa ou processamento em múltiplas etapas
        - Solucionável com no máximo 400 tokens de saída
        - NÃO inclua nenhum campo além dos especificados no formato de saída

        Aqui está um exemplo de entrada de amostra com uma saída ideal:
        <sample_input>
        <sample_task_description>
        Extrair tópicos de um trecho de texto
        </sample_task_description>
        <sample_specific_idea>
        Testando com um texto que contém múltiplos tópicos e sub-tópicos aninhados (por exemplo, um trecho sobre energia renovável que abrange economia da energia solar, tecnologia de turbinas eólicas e implicações políticas simultaneamente)
        </sample_specific_idea>

        <sample_allowed_input_keys>
        "content"
        </sample_allowed_input_keys>
        </sample_input>
        <ideal_output>
        ```json
        {
            "prompt_inputs": {
                "content": "A transição para energia renovável abrange inúmeras dimensões interdependentes. A tecnologia fotovoltaica solar viu reduções dramáticas de custos, com a eficiência dos painéis melhorando 24% desde 2010, enquanto os custos de fabricação diminuíram 89%, tornando-a economicamente competitiva com os combustíveis fósseis em muitos mercados. Paralelamente, a energia eólica evoluiu por meio de designs inovadores de turbinas com pás de compósito de fibra de carbono e sistemas de controle avançados que aumentam a captura de energia em 35% em condições de vento fraco."
            },
            "solution_criteria": [
                "Inclui todos os tópicos mencionados"   
            ]
        }
        ```
        </ideal_output>
        Este é o resultado ideal porque os critérios de solução são concisos e não solicitam nada fora do escopo da descrição da tarefa.
        """

        system_prompt = "Você é um criador de casos de teste especializado em projetar cenários de avaliação."

        rendered_prompt = self.render(
            dedent(prompt),
            {
                "allowed_keys": allowed_keys,
                "task_description": task_description,
                "idea": idea,
                "example_prompt_inputs": example_prompt_inputs,
            },
        )

        messages = []
        add_user_message(messages, rendered_prompt)
        add_assistant_message(messages, "```json")
        text = chat(
            messages,
            stop_sequences=["```"],
            system=system_prompt,
            temperature=0.7,
        )

        test_case = json.loads(text)
        test_case["task_description"] = task_description
        test_case["scenario"] = idea

        return test_case

    def generate_dataset(
        self,
        task_description,
        prompt_inputs_spec={},
        num_cases=1,
        output_file="dataset.json",
    ):
        """Generate test dataset based on task description and save to file"""
        ideas = self.generate_unique_ideas(
            task_description, prompt_inputs_spec, num_cases
        )

        dataset = []
        completed = 0
        total = len(ideas)
        last_reported_percentage = 0

        with concurrent.futures.ThreadPoolExecutor(
            max_workers=self.max_concurrent_tasks
        ) as executor:
            future_to_idea = {
                executor.submit(
                    self.generate_test_case,
                    task_description,
                    idea,
                    prompt_inputs_spec,
                ): idea
                for idea in ideas
            }

            for future in concurrent.futures.as_completed(future_to_idea):
                try:
                    result = future.result()
                    completed += 1
                    current_percentage = int((completed / total) * 100)
                    milestone_percentage = (current_percentage // 20) * 20

                    if milestone_percentage > last_reported_percentage:
                        print(f"Generated {completed}/{total} test cases")
                        last_reported_percentage = milestone_percentage

                    dataset.append(result)
                except Exception as e:
                    print(f"Error generating test case: {e}")

        with open(output_file, "w") as f:
            json.dump(dataset, f, indent=2)

        return dataset

    def grade_output(self, test_case, output, extra_criteria):
        """Grade the output of a test case using the model"""

        prompt_inputs = ""
        for key, value in test_case["prompt_inputs"].items():
            val = value.replace("\n", "\\n")
            prompt_inputs += f'"{key}":"{val}",\n'

        extra_criteria_section = ""
        if extra_criteria:
            extra_criteria_template = """
            Requisitos obrigatórios - QUALQUER VIOLAÇÃO SIGNIFICA FALHA AUTOMÁTICA (pontuação 3 ou inferior):
            <extra_important_criteria>
            {extra_criteria}
            </extra_important_criteria>
            """
            extra_criteria_section = self.render(
                dedent(extra_criteria_template),
                {"extra_criteria": extra_criteria},
            )

        eval_template = """
        Sua tarefa é avaliar a seguinte solução gerada por IA com RIGOR EXTREMO.

        Descrição original da tarefa:
        <task_description>
        {task_description}
        </task_description>

        Entradas originais da tarefa:
        <task_inputs>
        {{ {prompt_inputs} }}
        </task_inputs>

        Solução a ser avaliada:
        <solution>
        {output}
        </solution>

        Critérios que você deve usar para avaliar a solução:
        <criteria>
        {solution_criteria}
        </criteria>

        {extra_criteria_section}

        Diretrizes de Pontuação:
        * Pontuação 1-3: A solução não atende a um ou mais requisitos OBRIGATÓRIOS
        * Pontuação 4-6: A solução atende a todos os requisitos obrigatórios, mas apresenta deficiências significativas nos critérios secundários
        * Pontuação 7-8: A solução atende a todos os requisitos obrigatórios e à maioria dos critérios secundários, com pequenas falhas
        * Pontuação 9-10: A solução atende a todos os requisitos obrigatórios e secundários

        INSTRUÇÕES IMPORTANTES DE PONTUAÇÃO:
        * Avalie a saída com base APENAS nos critérios listados. Não adicione seus próprios requisitos extras.
        * Se uma solução atender a todos os critérios obrigatórios e secundários, dê a ela 10
        * Não reclame que a solução "apenas" atende aos critérios obrigatórios e secundários. As soluções não devem ir além - devem atender exatamente aos critérios listados.
        * QUALQUER violação de um requisito obrigatório DEVE resultar em uma pontuação de 3 ou inferior
        * A escala completa de 1 a 10 deve ser utilizada - não hesite em dar pontuações baixas quando necessário

        Formato de Saída
        Forneça sua avaliação como um objeto JSON estruturado com os seguintes campos, nesta ordem específica:
        - "strengths": Um array de 1-3 pontos fortes principais
        - "weaknesses": Um array de 1-3 áreas principais para melhoria
        - "reasoning": Uma explicação concisa da sua avaliação geral
        - "score": Um número entre 1-10

        Responda com JSON. Mantenha sua resposta concisa e direta.
        Exemplo de formato de resposta:
        {{
            "strengths": string[],
            "weaknesses": string[],
            "reasoning": string,
            "score": number
        }}
        """

        eval_prompt = self.render(
            dedent(eval_template),
            {
                "task_description": test_case["task_description"],
                "prompt_inputs": prompt_inputs,
                "output": output,
                "solution_criteria": "\n".join(test_case["solution_criteria"]),
                "extra_criteria_section": extra_criteria_section,
            },
        )

        messages = []
        add_user_message(messages, eval_prompt)
        add_assistant_message(messages, "```json")
        eval_text = chat(
            messages,
            stop_sequences=["```"],
            temperature=0.0,
        )
        return json.loads(eval_text)

    def run_test_case(self, test_case, run_prompt_function, extra_criteria=None):
        """Run a test case and grade the result"""
        output = run_prompt_function(test_case["prompt_inputs"])

        model_grade = self.grade_output(test_case, output, extra_criteria)
        model_score = model_grade["score"]
        reasoning = model_grade["reasoning"]

        return {
            "output": output,
            "test_case": test_case,
            "score": model_score,
            "reasoning": reasoning,
        }

    def run_evaluation(
        self,
        run_prompt_function,
        dataset_file,
        extra_criteria=None,
        json_output_file="output.json",
        html_output_file="output.html",
    ):
        """Run evaluation on all test cases in the dataset"""
        with open(dataset_file, "r") as f:
            dataset = json.load(f)

        results = []
        completed = 0
        total = len(dataset)
        last_reported_percentage = 0

        with concurrent.futures.ThreadPoolExecutor(
            max_workers=self.max_concurrent_tasks
        ) as executor:
            future_to_test_case = {
                executor.submit(
                    self.run_test_case,
                    test_case,
                    run_prompt_function,
                    extra_criteria,
                ): test_case
                for test_case in dataset
            }

            for future in concurrent.futures.as_completed(future_to_test_case):
                result = future.result()
                completed += 1
                current_percentage = int((completed / total) * 100)
                milestone_percentage = (current_percentage // 20) * 20

                if milestone_percentage > last_reported_percentage:
                    print(f"Graded {completed}/{total} test cases")
                    last_reported_percentage = milestone_percentage
                results.append(result)

        average_score = mean([result["score"] for result in results])
        print(f"Average score: {average_score}")

        with open(json_output_file, "w") as f:
            json.dump(results, f, indent=2)

        html = generate_prompt_evaluation_report(results)
        with open(html_output_file, "w", encoding="utf-8") as f:
            f.write(html)

        return results

In [13]:
# Create an instance of PromptEvaluator
# Increase `max_concurrent_tasks` for greater concurrency, but beware of rate limit errors!
evaluator = PromptEvaluator(max_concurrent_tasks=1)

In [14]:
dataset = evaluator.generate_dataset(
    # Describe the purpose or goal of the prompt you're trying to test
    task_description=" Você é personal trainer, seu trabalho é criar planos de treino personalizados para seus clientes com base nas informações que eles fornecem sobre seus objetivos, nível de condicionamento físico, restrições e preferências. Você deve usar as informações fornecidas para criar um plano de treino que seja seguro, eficaz e adaptado às necessidades individuais de cada cliente. ",
    # Describe the different inputs that your prompt requires
    prompt_inputs_spec={
        "Altura": "A altura do cliente em centímetros",
        "Peso": "O peso do cliente em kg",
        "Idade": "A idade do cliente em anos",
        "Objetivo": "O objetivo do cliente (ex: perder peso, ganhar massa muscular) "
    },
    # Where to write the generated dataset
    output_file="dataset.json",
    # Number of test cases to generate (recommend keeping this low if you're getting rate limit errors)
    num_cases=3,
)

Generated 1/3 test cases
Generated 2/3 test cases
Generated 3/3 test cases


In [15]:
# Define and run the prompt you want to evaluate, returning the raw model output
# This function is executed once for each test case
def run_prompt(prompt_inputs):
    prompt = f"""
    Quais exercícios você recomendaria para um cliente com as seguintes características?
    - Altura: {prompt_inputs["Altura"]} cm
    - Peso: {prompt_inputs["Peso"]} kg
    - Idade: {prompt_inputs["Idade"]} anos
    - Objetivo: {prompt_inputs["Objetivo"]} 
    """

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

In [16]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt, dataset_file="dataset.json",extra_criteria="a solução deve recomendar exercícios que sejam seguros e apropriados para a idade, peso e objetivo do cliente"
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 7
